# How many T4 GPUs do 500 users need, and what does it cost?

This notebook turns measured serving numbers into a capacity plan. It runs on CPU only. No GPU is needed.

The data is measured, not modeled. It comes from vLLM sweeps on Kaggle Tesla T4 GPUs. Three configs ran at concurrency 1, 4, 16 and 64:

- `1.5b-fp16-1gpu`: Qwen2.5-1.5B in fp16 on one T4. Source notebook: `vllm-t4-serving-benchmark`.
- `7b-fp16-tp2`: Qwen2.5-7B in fp16, split across two T4s with tensor parallelism. Source notebook: `vllm-t4-7b-tensor-parallel-vs-awq`.
- `7b-awq-1gpu`: Qwen2.5-7B in 4-bit AWQ on one T4. Source notebook: `vllm-t4-7b-tensor-parallel-vs-awq`.

Terms used here:

- **SLA** (service level agreement): the latency limit we promise to users.
- **TTFT**: time to first token. **TPOT**: time per output token after the first.
- **p99**: the value that 99 percent of requests stay below.
- **Concurrency**: requests in flight at the same moment.

## Predict first

Before you run any cell, write down a guess:

1. How many replicas of the 7B AWQ model will 500 users need?
2. Which of the three configs will be the cheapest per million output tokens?
3. Will two T4s with tensor parallelism need fewer or more GPUs than AWQ replicas on one T4 each?

Keep your guess. Compare it with the results at the end.

## 1. The measured data

The CSV text is embedded here, so the notebook runs on Kaggle without access to any repo.

In [ ]:
import io, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV = """experiment,config,model,gpus,precision,concurrency,requests,ttft_median_ms,ttft_p99_ms,tpot_median_ms,output_tok_s,requests_s
vllm-t4-serving-benchmark,1.5b-fp16-1gpu,Qwen/Qwen2.5-1.5B-Instruct,1,fp16,1,16,115,979,15.7,59,0.46
vllm-t4-serving-benchmark,1.5b-fp16-1gpu,Qwen/Qwen2.5-1.5B-Instruct,1,fp16,4,16,104,199,17.3,220,1.72
vllm-t4-serving-benchmark,1.5b-fp16-1gpu,Qwen/Qwen2.5-1.5B-Instruct,1,fp16,16,64,754,1734,28.5,463,3.62
vllm-t4-serving-benchmark,1.5b-fp16-1gpu,Qwen/Qwen2.5-1.5B-Instruct,1,fp16,64,256,1461,6878,96.1,650,5.08
vllm-t4-7b-tensor-parallel-vs-awq,7b-fp16-tp2,Qwen/Qwen2.5-7B-Instruct,2,fp16,1,16,263,339,32.2,28,0.22
vllm-t4-7b-tensor-parallel-vs-awq,7b-fp16-tp2,Qwen/Qwen2.5-7B-Instruct,2,fp16,4,16,228,310,36.2,106,0.83
vllm-t4-7b-tensor-parallel-vs-awq,7b-fp16-tp2,Qwen/Qwen2.5-7B-Instruct,2,fp16,16,64,1509,3750,55.7,231,1.80
vllm-t4-7b-tensor-parallel-vs-awq,7b-fp16-tp2,Qwen/Qwen2.5-7B-Instruct,2,fp16,64,256,2744,13780,170.4,371,2.90
vllm-t4-7b-tensor-parallel-vs-awq,7b-awq-1gpu,Qwen/Qwen2.5-7B-Instruct-AWQ,1,int4-awq,1,16,423,493,25.0,35,0.28
vllm-t4-7b-tensor-parallel-vs-awq,7b-awq-1gpu,Qwen/Qwen2.5-7B-Instruct-AWQ,1,int4-awq,4,16,285,397,27.3,136,1.06
vllm-t4-7b-tensor-parallel-vs-awq,7b-awq-1gpu,Qwen/Qwen2.5-7B-Instruct-AWQ,1,int4-awq,16,64,3016,6719,58.9,192,1.50
vllm-t4-7b-tensor-parallel-vs-awq,7b-awq-1gpu,Qwen/Qwen2.5-7B-Instruct-AWQ,1,int4-awq,64,256,5043,27121,264.1,241,1.88
"""
df = pd.read_csv(io.StringIO(CSV))
df.drop(columns=["experiment", "model", "requests"])

## 2. SLA knobs and the highest safe concurrency

Set the two limits. Then find, for each config, the highest concurrency that meets both.

Between two measured points we interpolate on log2(concurrency). We never go past the last measured point (64). If even concurrency 1 fails, the answer is 0.

In [ ]:
TTFT_P99_MS = 2000      # SLA: p99 time to first token
TPOT_MEDIAN_MS = 50     # SLA: median time per output token

def sla_log2(x, y, limit):
    """Highest log2(concurrency) where metric y stays <= limit. Metrics rise with load."""
    if y[0] > limit:
        return None                       # fails even at concurrency 1
    for i in range(1, len(x)):
        if y[i] > limit:                  # crossing is inside this segment
            return x[i-1] + (limit - y[i-1]) / (y[i] - y[i-1]) * (x[i] - x[i-1])
    return x[-1]                          # never fails: cap at last measured point

rows = []
for name, g in df.groupby("config", sort=False):
    g = g.sort_values("concurrency")
    x = np.log2(g.concurrency.values)
    xs = [sla_log2(x, g.ttft_p99_ms.values, TTFT_P99_MS), sla_log2(x, g.tpot_median_ms.values, TPOT_MEDIAN_MS)]
    c = 0.0 if None in xs else 2 ** min(xs)
    at = lambda col: float(np.interp(math.log2(c), x, g[col].values)) if c else 0.0
    rows.append(dict(config=name, gpus=int(g.gpus.iloc[0]), sla_concurrency=c,
                     ttft_median_ms=at("ttft_median_ms"), tpot_median_ms=at("tpot_median_ms"),
                     output_tok_s=at("output_tok_s")))
sla = pd.DataFrame(rows).set_index("config")
sla.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col, limit, label in [(axes[0], "tpot_median_ms", TPOT_MEDIAN_MS, "TPOT median (ms)"),
                              (axes[1], "ttft_p99_ms", TTFT_P99_MS, "TTFT p99 (ms)")]:
    for name, g in df.groupby("config", sort=False):
        ax.plot(g.concurrency, g[col], marker="o", label=name)
    ax.axhline(limit, color="k", ls="--", label="SLA")
    ax.set_xscale("log", base=2); ax.set_yscale("log")
    ax.set_xlabel("Concurrency (requests in flight)"); ax.set_ylabel(label); ax.grid(alpha=.3)
axes[0].legend(); plt.tight_layout(); plt.show()

## 3. Little's law: how many requests are in flight?

**Little's law** links three numbers in a stable system: the average number of requests in flight equals the arrival rate times the time each request takes.

Our users are a **closed system**: each user sends a request, waits for the answer, thinks, then sends the next one. So a user's cycle is think time plus request time.

- `ACTIVE_USERS` (N): users who use the service at the same time.
- `THINK_TIME_S` (Z): seconds a user waits between an answer and the next request.
- `OUTPUT_TOKENS`: tokens in each answer.
- Request time W: TTFT plus `OUTPUT_TOKENS` times TPOT, taken at the SLA point. We use median TTFT as the typical request.
- Arrival rate lambda = N / (Z + W): requests per second the whole crowd sends.
- Concurrent requests L = lambda x W: requests in flight on average.

In [ ]:
ACTIVE_USERS = 500
THINK_TIME_S = 60
OUTPUT_TOKENS = 128

sla["W_s"] = (sla.ttft_median_ms + OUTPUT_TOKENS * sla.tpot_median_ms) / 1000
sla["lambda_rps"] = ACTIVE_USERS / (THINK_TIME_S + sla.W_s)
sla["L_concurrent"] = sla.lambda_rps * sla.W_s
sla[["W_s", "lambda_rps", "L_concurrent"]].round(3)

## 4. Replicas and GPUs

A replica is one copy of the model server. One replica can hold at most the SLA concurrency from section 2. So replicas = ceil(L / that number). GPUs = replicas times GPUs per replica.

Then a throughput check: the fleet must produce tokens as fast as users ask for them.

In [ ]:
sla["replicas"] = np.ceil(sla.L_concurrent / sla.sla_concurrency).astype(int)
sla["total_gpus"] = sla.replicas * sla.gpus
sla["tokens_needed_s"] = sla.lambda_rps * OUTPUT_TOKENS
sla["tokens_capacity_s"] = sla.replicas * sla.output_tok_s
sla["throughput_ok"] = sla.tokens_needed_s <= sla.tokens_capacity_s
for name, r in sla.iterrows():
    print(f"{name}: {r.replicas} replicas, {r.total_gpus} GPUs | need {r.tokens_needed_s:.0f} tok/s, "
          f"have {r.tokens_capacity_s:.0f} tok/s -> {'PASS' if r.throughput_ok else 'FAIL'}")

## 5. Cost

The price below is an **assumption, not a quote**. Set the on-demand price of your own provider.

In [ ]:
PRICE_PER_GPU_HOUR = 0.50   # assumption in USD, not a quote: set your provider's price
HOURS_PER_MONTH = 730

sla["usd_per_1m_tokens"] = sla.gpus * PRICE_PER_GPU_HOUR / (sla.output_tok_s * 3600) * 1e6
sla["usd_per_month"] = sla.total_gpus * PRICE_PER_GPU_HOUR * HOURS_PER_MONTH
sla["usd_per_user_month"] = sla.usd_per_month / ACTIVE_USERS
sla[["replicas", "total_gpus", "usd_per_1m_tokens", "usd_per_month", "usd_per_user_month"]].round(2)

## 6. Horizontal vs vertical scaling

The two 7B options give the same answers and the same quality target. They scale in different ways:

- **Vertical**: make one replica bigger. TP=2 spreads one model over 2 GPUs.
- **Horizontal**: add more copies. AWQ fits the model on 1 GPU, so we add replicas.

In [ ]:
v, h = sla.loc["7b-fp16-tp2"], sla.loc["7b-awq-1gpu"]
fewer = "vertical (TP=2)" if v.total_gpus < h.total_gpus else "horizontal (AWQ)" if h.total_gpus < v.total_gpus else "neither (tie)"
print(f"Vertical TP=2:    {v.replicas} replicas x {v.gpus} GPUs = {v.total_gpus} GPUs")
print(f"Horizontal AWQ:   {h.replicas} replicas x {h.gpus} GPU  = {h.total_gpus} GPUs")
print(f"Fewer GPUs: {fewer}. Per replica, TP=2 holds {v.sla_concurrency:.1f} requests within the SLA "
      f"against {h.sla_concurrency:.1f} for AWQ, and gives {v.output_tok_s:.0f} vs {h.output_tok_s:.0f} tok/s.")

## 7. Monitoring drill (synthetic)

Sizing is not the end. You must also detect when the SLA breaks. This drill uses made-up data, built around the AWQ TPOT at its SLA point.

- 2 hours of TPOT samples, one every 10 seconds, with noise.
- At minute 60 a traffic spike doubles TPOT.
- Alert rule: the rolling 5-minute p99 TPOT is above the SLA for 3 checks in a row. Checks run once per minute.
- Mitigation: scale out. It takes effect 10 minutes after the alert and returns TPOT to baseline.
- **MTTD** (mean time to detect): alert time minus incident start.
- **MTTR** (mean time to recover): recovery time minus incident start. With one incident it is just that one value.

In [ ]:
rng = np.random.default_rng(0)
base = sla.loc["7b-awq-1gpu", "tpot_median_ms"]
t = np.arange(0, 7200, 10)                          # seconds
INCIDENT_S, FIX_DELAY_S, CHECK_S, WINDOW_S, NEED = 3600, 600, 60, 300, 3

def run(t, fix_at=None):
    tpot = base + rng.normal(0, 0.08 * base, t.size)
    spike = (t >= INCIDENT_S) & ((t < fix_at) if fix_at else True)
    return np.where(spike, 2 * tpot, tpot)

def first_alert(t, tpot):
    streak = 0
    for now in range(WINDOW_S, int(t[-1]) + 1, CHECK_S):
        w = tpot[(t > now - WINDOW_S) & (t <= now)]
        streak = streak + 1 if np.percentile(w, 99) > TPOT_MEDIAN_MS else 0
        if streak == NEED:
            return now
alert_s = first_alert(t, run(t))                    # no mitigation: find the alert
recover_s = alert_s + FIX_DELAY_S
rng = np.random.default_rng(0)                      # same noise, now with the fix applied
tpot = run(t, fix_at=recover_s)
print(f"Baseline TPOT {base:.1f} ms | SLA {TPOT_MEDIAN_MS} ms")
print(f"MTTD = {(alert_s - INCIDENT_S) / 60:.1f} min | MTTR = {(recover_s - INCIDENT_S) / 60:.1f} min")

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(t / 60, tpot, lw=.7, label="TPOT sample")
plt.axhline(TPOT_MEDIAN_MS, color="k", ls="--", label="SLA")
for s, lab, c in [(INCIDENT_S, "incident start", "r"), (alert_s, "alert", "orange"), (recover_s, "recovery", "g")]:
    plt.axvline(s / 60, color=c, ls=":", label=lab)
plt.xlabel("Time (minutes)"); plt.ylabel("TPOT (ms)"); plt.legend(); plt.grid(alpha=.3); plt.show()

## Exam rules this proves

Compare these with your first guesses.

1. Size from the SLA, not from the peak numbers. Use the highest load that still meets the latency limit, then divide the in-flight requests by it. (sizing.3, inference.6)
2. Little's law gives the in-flight requests: users divided by (think time plus request time), times request time. (sizing.3)
3. Check throughput as well as latency. The fleet must make tokens at least as fast as users ask for them. (sizing.3, inference.6)
4. Compare vertical and horizontal scaling by total GPUs and by cost per token, not by one number. Lower precision such as AWQ changes both memory use and cost. (sizing.3, sizing.5, inference.8)
5. Cost follows model size, tokens, traffic, hardware and latency target. Alert on p99 latency, then measure MTTD and MTTR. (inference.8, sizing.2)